In [ ]:
## Import modules
import os,sys
import numpy as np
import geopandas as gpd
import cftime 
import gc
import glob
import json 
import xarray as xr
import datetime
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numba 
import concurrent.futures
from pathlib import Path

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger 
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0,cmct_dir)
from cmct.time_utils import check_datarange
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.json_to_netcdf import *
from cmct.shapefile_utils import *

from cmct.gravimetry import *
from cmct.projection import *

# Force initial garbage collection
gc.collect()


In [ ]:
def process_single_model(model_filename, gsfc, start_year, end_year, output_dir, filename_prefix, interpolation_method='nearest'):
    """
    Process a single model file through the complete comparison workflow.
    
    Parameters
    ----------
    model_filename : str
        Path to model netCDF file
    gsfc : GSFCcalving
        Observation data object
    start_year : int
        Start year for comparison
    end_year : int
        End year for comparison
    output_dir : str
        Output directory path
    filename_prefix : str
        Prefix for output filename
    interpolation_method : str
        Interpolation method to use
        
    Returns
    -------
    tuple
        (output_path, success_flag, error_message)
    """
    try:
        print(f"\n{'='*60}")
        print(f"Processing: {os.path.basename(model_filename)}")
        print(f"{'='*60}")
        
        # Load and validate model data
        model_res = load_and_validate_model_data(model_filename, start_year, end_year)
        
        # Perform interpolation and comparison
        residuals, gsfc_ds_residual, analyses = perform_interpolation_and_comparison(
            gsfc, model_res, start_year, end_year, interpolation_method
        )
        
        # Save results to NetCDF
        output_path = save_results_to_netcdf(
            gsfc_ds_residual, analyses, model_filename, output_dir, filename_prefix
        )
        
        # Clean up memory
        del model_res, residuals, gsfc_ds_residual
        gc.collect()
        
        print(f"Successfully processed: {os.path.basename(model_filename)}")
        return output_path, True, None
        
    except Exception as e:
        error_msg = f"Error processing {model_filename}: {str(e)}"
        print(essrror_msg)
        logging.error(error_msg)
        return None, False, str(e)

In [ ]:
def save_results_to_netcdf(gsfc_ds_residual, analyses, model_filename, output_dir, filename_prefix):
    """
    Save comparison results to NetCDF file.
    
    Parameters
    ----------
    gsfc_ds_residual : xarray.Dataset
        Dataset containing residual comparison results
    analyses : list
        List of analysis dictionaries for each year
    model_filename : str
        Original model filename for naming output
    output_dir : str
        Output directory path
    filename_prefix : str
        Prefix for output filename
        
    Returns
    -------
    str
        Path to saved NetCDF file
    """
    # Extract model name from filename
    model_name = os.path.splitext(os.path.basename(model_filename))[0]
    
    # Create output filename
    output_filename = f"{filename_prefix}_{model_name}.nc"
    output_path = os.path.join(output_dir, output_filename)
    
    # Add analysis metadata to dataset attributes
    for i, analysis in enumerate(analyses):
        year = gsfc_ds_residual.year.values[i] if i < len(gsfc_ds_residual.year.values) else None
        if year is not None:
            gsfc_ds_residual.attrs[f'year_{year}_avg_residual'] = analysis['ABS_AVG_RESIDUAL']
            gsfc_ds_residual.attrs[f'year_{year}_rms_residual'] = analysis['RMS_RESIDUAL']
            gsfc_ds_residual.attrs[f'year_{year}_valid_points'] = analysis['VALID_POINTS']
            gsfc_ds_residual.attrs[f'year_{year}_sum_residual'] = analysis['SUM_RESIDUAL']
    
    # Add global metadata
    gsfc_ds_residual.attrs['model_file'] = model_filename
    gsfc_ds_residual.attrs['processing_date'] = datetime.datetime.now().isoformat()
    gsfc_ds_residual.attrs['description'] = 'Calving comparison results between observation and model data'
    
    # Save to NetCDF
    print(f"Saving results to: {output_path}")
    gsfc_ds_residual.to_netcdf(output_path, mode='w', format='NETCDF4')
    
    return output_path

In [ ]:
# Configuration for ensemble comparison
# Ice sheet location
loc = 'GIS' # 'GIS' or 'AIS'

# Set time range for comparison
start_year = 2006
end_year = 2010

# Set the observation data dir path
obs_filename = cmct_dir + '/data/calving/observed_icemask_ismip_annual.nc'

# To use aggregation functions for basin 
basin_aggregation = True # IMPORTANT
basin_shapes = cmct_dir + '/data/ne_10m_coastline/ne_10m_coastline.shp'

# Set the Model Data dir path (ensemble files)
if loc == "GIS":
    # Greenland
    mod_filename_template = cmct_dir + '/test/calving/gris*.nc'
elif loc == "AIS":    
    # Antarctica
    mod_filename_template = cmct_dir + '/test/calving/ais*.nc'

# Output configuration
output_dir = cmct_dir + '/notebooks/Calving/ensemble_results/'
filetype = 'netcdf' # netcdf or json or None
filename_prefix = 'calving_comparison_ensemble'

# Processing configuration
interpolation_method = 'nearest' # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = 'mean' # 'mean', 'RMS'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")


# Loading Observation Data 


In [ ]:
print(obs_filename)
gsfc = load_gsfc_calving(obs_filename)

# Comparing Ensemble Data

In [ ]:
nc_filenames = glob.glob(mod_filename_template)
if not nc_filenames:
    raise FileNotFoundError(f"No model files found matching template: {mod_filename_template}")
else:
    print(f"Found {len(nc_filenames)} model files for ensemble processing:")
    for i, filename in enumerate(nc_filenames, 1):
        print(f"  {i}. {os.path.basename(filename)}")

print(f"\nConfiguration Summary:")
print(f"  Observation file: {os.path.basename(obs_filename)}")
print(f"  Model file pattern: {mod_filename_template}")
print(f"  Time range: {start_year}-{end_year}")
print(f"  Output directory: {output_dir}")
print(f"  Interpolation method: {interpolation_method}")
print(f"  Basin aggregation: {basin_aggregation}")

# Chunking Process and assign to workers






In [ ]:
def load_and_validate_observation_data(obs_filename):
    """
    Load and validate observation data.
    
    Parameters
    ----------
    obs_filename : str
        Path to observation netCDF file
        
    Returns
    -------
    GSFCcalving
        Loaded observation data object
    """
    if not os.path.exists(obs_filename):
        raise FileNotFoundError(f"Observation file not found: {obs_filename}")
    
    print(f"Loading observation data: {obs_filename}")
    gsfc = load_gsfc_calving(obs_filename)
    
    # Standardize time format
    gsfc.ds["time"] = gsfc.time
    
    return gsfc

# Loop through each file 
for nc_filename in nc_filenames:
    
    
    print(f"\nProcessing:{nc_filename}")

    # Load model data
    model_res = load_model_calving(nc_filename)
    time_var = model_res['time']
    
    # Convert start/end comparison times to fractional year
    calendar_type = time_var.to_index().calendar
    start_date_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    end_date_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
    
    # Adjust day to be 30 ( to avoid error if it's the 31st day in a 360_day calendar)
    start_date_cftime = cftime.datetime(start_date_dt.year, start_date_dt.month, min(start_date_dt.day, 30), calendar=calendar_type)
    end_date_cftime = cftime.datetime(end_date_dt.year, end_date_dt.month, min(end_date_dt.day, 30), calendar=calendar_type)
  
    # Check the selcted dates are within the range of model data
    check_datarange(time_var,start_date_cftime, end_date_cftime)
    
    # Put model into mascon space and calulate mass change of model data
    mass_change_mod_trim, mass_change_mod = transformToGeodetic(gsfc, gis_ds, start_date_cftime, end_date_cftime, rho_ice, rho_water, polar_stereographic)
    
    # Calculate mass change of model and observation data
    mass_change_delta = mass_change_mod_trim-mass_change_obs
    
    # Write result to nc file
    output_filename = os.path.splitext(os.path.basename(nc_filename))[0] + '_mascon_comp'
    output_netcdf_filename = output_netcdf_filepath + output_filename + '.nc'
    write_to_netcdf(mass_change_obs, mass_change_delta, pa mass_change_mod_trim, gsfc, I_, start_date_cftime, end_date_cftime, output_netcdf_filename)


In [ ]:
def load_and_validate_model_data(model_filename, start_year, end_year):
    """
    Load and validate a single model data file.
    
    Parameters
    ----------
    model_filename : str
        Path to model netCDF file
    start_year : int
        Start year for comparison
    end_year : int
        End year for comparison
        
    Returns
    -------
    Modelcalving
        Loaded model data object
    """
    if not os.path.exists(model_filename):
        raise FileNotFoundError(f"Model file not found: {model_filename}")
    
    print(f"Loading model data: {model_filename}")
    model_res = load_model_calving(model_filename)
    
    # Standardize time format
    model_res.ds["time"] = [dt.year for dt in model_res.time.values]
    
    return model_res